In [1]:
%load_ext autoreload
%autoreload 2

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

Note: you may need to restart the kernel to use updated packages.


In [3]:
from src.data import DATA_DIR, load_acs
print (DATA_DIR)

X, y, group = load_acs(2014)
print(X.shape, y.shape, group.shape)

X.head()

/Users/nguyenngochongnhi/Desktop/UTA/Special Topic in AI/Project/fairness-over-time/data/raw
(127039, 10) (127039, 1) (127039, 1)


,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P
0,53.0,1.0,14.0,1.0,5700.0,4.0,1.0,40.0,2.0,1.0
1,68.0,5.0,22.0,1.0,530.0,48.0,0.0,40.0,1.0,1.0
2,64.0,3.0,22.0,1.0,2310.0,48.0,1.0,20.0,2.0,1.0
3,54.0,1.0,21.0,1.0,800.0,48.0,0.0,50.0,1.0,1.0
4,51.0,7.0,18.0,1.0,4700.0,39.0,1.0,40.0,2.0,1.0


## Data dictionary: ACSIncome (Texas, 2014)

Each row is one survey respondent. ACSIncome keeps only people aged 17+, with income over $100, who usually worked at least 1 hour per week, and with a survey weight of at least 1.

### Features (`X`)

| Column | Meaning | Type | Codes |
|---|---|---|---|
| `AGEP` | Age in years | Numeric | 17–92 |
| `COW` | Class of worker (who the person works for) | Nominal | 1 private for-profit · 2 private nonprofit · 3 local gov · 4 state gov · 5 federal gov · 6 self-employed, not incorporated · 7 self-employed, incorporated · 8 unpaid family business |
| `SCHL` | Highest education completed | Ordinal | 1 no schooling … 16 HS diploma · 17 GED · 18–19 some college · 20 associate's · 21 bachelor's · 22 master's · 23 professional degree · 24 doctorate |
| `MAR` | Marital status | Nominal | 1 married · 2 widowed · 3 divorced · 4 separated · 5 never married |
| `OCCP` | Occupation | Nominal | 478 codes in 2014 (codes change in 2018) |
| `POBP` | Place of birth | Nominal | 1–56 US states (48 = Texas) · 60–78 US territories · 100+ other countries |
| `RELP` | Relationship to householder | Nominal | 0 householder · 1 spouse · 2+ child, parent, roommate, etc. |
| `WKHP` | Hours usually worked per week | Numeric | 1–99 (99 = 99 or more) |
| `SEX` | Sex (sensitive attribute) | Nominal | 1 male · 2 female |
| `RAC1P` | Race (sensitive attribute) | Nominal | 1 White · 2 Black · 3 American Indian · 4 Alaska Native · 5 AIAN, tribes specified · 6 Asian · 7 Native Hawaiian/Pacific Islander · 8 Some Other Race · 9 Two or More Races |

### Label (`y`)

| Column | Meaning |
|---|---|
| `PINCP` | True if total personal income (all sources) is over $50,000; False otherwise |

### Sensitive group (`group`)

| Column | Meaning |
|---|---|
| `RAC1P` | Copy of the race column, used for fairness analysis |

*Source: 2014 ACS PUMS Data Dictionary (U.S. Census Bureau).*

In [4]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 127039 entries, 0 to 127038
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   AGEP    127039 non-null  float64
 1   COW     127039 non-null  float64
 2   SCHL    127039 non-null  float64
 3   MAR     127039 non-null  float64
 4   OCCP    127039 non-null  float64
 5   POBP    127039 non-null  float64
 6   RELP    127039 non-null  float64
 7   WKHP    127039 non-null  float64
 8   SEX     127039 non-null  float64
 9   RAC1P   127039 non-null  float64
dtypes: float64(10)
memory usage: 9.7 MB


In [5]:
X.describe()

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P
count,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000
mean,42.690316,2.053275,18.073615,2.415565,4236.300301,86.972977,2.000464,39.755437,1.465715,1.840380
std,14.778244,1.807224,3.757306,1.749398,2580.129698,100.190467,4.052481,13.325580,0.498825,2.063789
min,17.000000,1.000000,1.000000,1.000000,10.000000,1.000000,0.000000,1.000000,1.000000,1.000000
25%,30.000000,1.000000,16.000000,1.000000,2310.000000,47.000000,0.000000,36.000000,1.000000,1.000000
50%,43.000000,1.000000,19.000000,1.000000,4250.000000,48.000000,0.000000,40.000000,1.000000,1.000000
75%,54.000000,3.000000,21.000000,5.000000,5700.000000,48.000000,1.000000,45.000000,2.000000,1.000000
max,92.000000,8.000000,24.000000,5.000000,9830.000000,554.000000,17.000000,99.000000,2.000000,9.000000


In [6]:
print("Duplicate rows:", X.duplicated().sum())

Duplicate rows: 3714


In [7]:
y.value_counts()

PINCP
False    86850
True     40189
Name: count, dtype: int64

In [8]:
group.value_counts()

RAC1P
1        99527
2        12397
8         6294
6         5510
9         2560
3          457
5          192
7           96
4            6
Name: count, dtype: int64

In [9]:
y["PINCP"].groupby(X["SEX"]).mean()

SEX
1.0    0.393473
2.0    0.227875
Name: PINCP, dtype: float64

In [10]:
pd.crosstab(X["RAC1P"], X["SEX"])

SEX,1.0,2.0
RAC1P,,
1.0,53720,45807
2.0,5789,6608
3.0,237,220
4.0,2,4
5.0,110,82
6.0,2936,2574
7.0,52,44
8.0,3645,2649
9.0,1384,1176


In [11]:
import inspect
from folktables import acs
print(inspect.getsource(acs.adult_filter))

def adult_filter(data):
    """Mimic the filters in place for Adult data.

    Adult documentation notes: Extraction was done by Barry Becker from
    the 1994 Census database. A set of reasonably clean records was extracted
    using the following conditions:
    ((AAGE>16) && (AGI>100) && (AFNLWGT>1)&& (HRSWK>0))
    """
    df = data
    df = df[df['AGEP'] > 16]
    df = df[df['PINCP'] > 100]
    df = df[df['WKHP'] > 0]
    df = df[df['PWGTP'] >= 1]
    return df



In [85]:
numeric_cols = ['AGEP']
ordinal_cols = ['SCHL', 'WKHP']
nominal_cols = ['COW', 'MAR', 'OCCP', 'POBP', 'SEX', 'race_eth', 'RELP']

In [13]:
assert sorted(numeric_cols + ordinal_cols + nominal_cols) == sorted(X.columns)

In [14]:
X[nominal_cols].nunique()

COW        8
MAR        5
OCCP     478
POBP     211
SEX        2
RAC1P      9
RELP      18
dtype: int64

In [54]:
from src.data import split_data
train_idx, val_idx, test_idx = split_data(X, y)

print(len(train_idx), len(val_idx), len(test_idx))

76223 25408 25408


In [50]:
#verify
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(name, y.loc[idx, "PINCP"].mean())

train 0.3163612033113365
val 0.3163176952141058
test 0.31635705289672544


In [ ]:
#verify
print(set(train_idx) & set(val_idx), set(val_idx) & set(test_idx))   # both set()
a, _, _ = split_data(X, y, seed=7)
print(set(a) == set(train_idx))                                       # False

set() set()
False


In [60]:
#saving the split
from src.data import save_split, load_split

split_df = save_split(train_idx, val_idx, test_idx)
print(split_df["split"].value_counts())

# Round-trip check: does what we load back match what we saved?
tr, va, te = load_split()
print(set(tr) == set(train_idx), set(va) == set(val_idx), set(te) == set(test_idx)) 

split_df.head()

split
train    76223
val      25408
test     25408
Name: count, dtype: int64
True True True


,row_id,split
10169,0,val
11171,1,test
12286,2,train
26447,3,train
14337,4,val


In [73]:
#preprocessing

In [64]:
from folktables import ACSDataSource
source = ACSDataSource(survey_year="2014", horizon="1-Year", survey="person", root_dir=str(DATA_DIR))
raw = source.get_data(states=["TX"], download=False)

raw.head()

,RT,SERIALNO,SPORDER,PUMA,ST,ADJINC,PWGTP,AGEP,CIT,CITWP,...,pwgtp71,pwgtp72,pwgtp73,pwgtp74,pwgtp75,pwgtp76,pwgtp77,pwgtp78,pwgtp79,pwgtp80
0,P,3,1,3602,48,1008425,90,69,1,NaN,...,101,30,31,25,95,79,76,87,91,25
1,P,3,2,3602,48,1008425,82,53,1,NaN,...,81,27,24,21,98,67,69,93,101,21
2,P,3,3,3602,48,1008425,128,11,1,NaN,...,113,42,47,35,128,108,100,116,127,27
3,P,5,1,6603,48,1008425,112,68,1,NaN,...,114,99,140,31,110,111,109,178,100,205
4,P,5,2,6603,48,1008425,97,64,1,NaN,...,103,95,100,25,92,90,91,142,101,211


In [ ]:
filtered = acs.adult_filter(raw)
len(filtered)

127039

In [66]:
filtered = filtered.reset_index(drop=True)
(filtered["AGEP"].values == X["AGEP"].values).all()   # should be True 

np.True_

In [68]:
pd.crosstab(filtered["RAC1P"], filtered["HISP"] == 1)

HISP,False,True
RAC1P,,
1,31526,68001
2,295,12102
3,141,316
4,0,6
5,150,42
6,68,5442
7,21,75
8,6169,125
9,890,1670


In [70]:
filtered["is_hispanic"] = filtered["HISP"] != 1
y["PINCP"].groupby(filtered["is_hispanic"]).mean()

/var/folders/hh/vzwqsyz92pbdyc1q7kc30_k00000gn/T/ipykernel_80249/3402199326.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  filtered["is_hispanic"] = filtered["HISP"] != 1


is_hispanic
False    0.382187
True     0.169154
Name: PINCP, dtype: float64

In [71]:
(filtered["AGEP"].values == X["AGEP"].values).all()

np.True_

In [72]:
filtered.head()

,RT,SERIALNO,SPORDER,PUMA,ST,ADJINC,PWGTP,AGEP,CIT,CITWP,...,pwgtp72,pwgtp73,pwgtp74,pwgtp75,pwgtp76,pwgtp77,pwgtp78,pwgtp79,pwgtp80,is_hispanic
0,P,3,2,3602,48,1008425,82,53,1,NaN,...,27,24,21,98,67,69,93,101,21,False
1,P,5,1,6603,48,1008425,112,68,1,NaN,...,99,140,31,110,111,109,178,100,205,True
2,P,5,2,6603,48,1008425,97,64,1,NaN,...,95,100,25,92,90,91,142,101,211,True
3,P,48,1,3900,48,1008425,35,54,1,NaN,...,35,78,79,52,40,10,27,44,45,False
4,P,48,2,3900,48,1008425,29,51,1,NaN,...,30,75,48,36,20,7,35,45,43,False


In [76]:
from src.data import make_race_eth, load_acs

race_eth = make_race_eth(filtered)
print(race_eth.value_counts())

# One-row test: a White Hispanic person should be "Hispanic"
print(make_race_eth(pd.DataFrame({"RAC1P": [1], "HISP": [2]})))

race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64
0    Hispanic
Name: race_eth, dtype: str


In [77]:
X, y, meta = load_acs(2014)
print(X.shape, y.shape, meta.shape)
print(meta.head())
print(meta["race_eth"].value_counts())

(127039, 10) (127039, 1) (127039, 5)
   race_eth  SEX   income   ADJINC  PWGTP
0  NH White    2  36000.0  1008425     82
1  Hispanic    1  82000.0  1008425    112
2  Hispanic    2  55200.0  1008425     97
3  NH White    1  48000.0  1008425     35
4  NH White    2  75000.0  1008425     29
race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64


In [83]:
train_idx, val_idx, test_idx = split_data(y, meta)
print(len(train_idx), len(val_idx), len(test_idx))

# Strata check: how many strata, and the smallest?
strata = y["PINCP"].astype(str) + "_" + meta["race_eth"] + "_" + meta["SEX"].astype(str)
print(strata.nunique(), strata.value_counts().min())

# Positive rate per set
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(name, y.loc[idx, "PINCP"].mean())

# Overwrite the saved split
save_split(train_idx, val_idx, test_idx)
tr, va, te = load_split()
print(set(tr) == set(train_idx), set(va) == set(val_idx), set(te) == set(test_idx))

76223 25408 25408
20 216
train 0.3163480839116802
val 0.3163176952141058
test 0.3163964105793451
True True True


In [86]:
X, y, meta = load_acs(2014)
print(X.shape)                        # still (127039, 10)
print(X.columns.tolist())             # race_eth present, RAC1P gone
print(X["race_eth"].value_counts())   # same 5 counts as before

assert sorted(numeric_cols + ordinal_cols + nominal_cols) == sorted(X.columns)

(127039, 10)
['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'race_eth']
race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64


In [87]:
occ = X["OCCP"].value_counts()
pob = X["POBP"].value_counts()
for t in [10, 30, 50, 100]:
    print(t, "OCCP kept:", (occ >= t).sum(), "| POBP kept:", (pob >= t).sum())

10 OCCP kept: 441 | POBP kept: 162
30 OCCP kept: 362 | POBP kept: 118
50 OCCP kept: 309 | POBP kept: 97
100 OCCP kept: 225 | POBP kept: 72


In [88]:
occ_tr = X.loc[train_idx, "OCCP"].value_counts()
pob_tr = X.loc[train_idx, "POBP"].value_counts()
print("OCCP rows in rare codes:", occ_tr[occ_tr < 30].sum() / occ_tr.sum())
print("POBP rows in rare codes:", pob_tr[pob_tr < 30].sum() / pob_tr.sum())

OCCP rows in rare codes: 0.02744578408092046
POBP rows in rare codes: 0.013906563635648033


In [91]:
from src.preprocess import NUMERIC_COLS, ORDINAL_COLS, NOMINAL_COLS, build_preprocessor

pre = build_preprocessor()
Xt_train = pre.fit_transform(X.loc[train_idx])   # fit on TRAIN only
Xt_val = pre.transform(X.loc[val_idx])
Xt_test = pre.transform(X.loc[test_idx])

print(Xt_train.shape, Xt_val.shape, Xt_test.shape)
names = pre.get_feature_names_out()
print(len(names), [n for n in names if "infrequent" in n])

(76223, 456) (25408, 456) (25408, 456)
456 ['OCCP_infrequent_sklearn', 'POBP_infrequent_sklearn', 'RELP_infrequent_sklearn']


In [92]:
X.loc[train_idx, "RELP"].value_counts().sort_values().head()

RELP
14.0      5
8.0      77
3.0     181
11.0    300
6.0     386
Name: count, dtype: int64

In [95]:
from sklearn.metrics import accuracy_score, roc_auc_score
from src.models import make_lr, make_xgb, make_pipeline_for

y_train = y.loc[train_idx, "PINCP"].astype(int)
y_val = y.loc[val_idx, "PINCP"].astype(int)

for name, model in [("LR", make_lr()), ("XGB", make_xgb())]:
    pipe = make_pipeline_for(model)
    pipe.fit(X.loc[train_idx], y_train)
    proba = pipe.predict_proba(X.loc[val_idx])[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(name, "acc:", accuracy_score(y_val, pred), "AUC:", roc_auc_score(y_val, proba))

LR acc: 0.8230872166246851 AUC: 0.8883523326971859
XGB acc: 0.8282037153652393 AUC: 0.8956414681516556


In [96]:
#tuning
for depth in [4, 6, 8]:
    pipe = make_pipeline_for(make_xgb(max_depth=depth))
    pipe.fit(X.loc[train_idx], y_train)
    proba = pipe.predict_proba(X.loc[val_idx])[:, 1]
    print(depth, roc_auc_score(y_val, proba))

4 0.8922078530541567
6 0.8956414681516556
8 0.8963892688561103


In [98]:
from src.fairness import group_metrics

pipe = make_pipeline_for(make_xgb())
pipe.fit(X.loc[train_idx], y_train)
pred_val = pipe.predict(X.loc[val_idx])
sens_val = meta.loc[val_idx]

print(group_metrics(y_val, pred_val, sens_val["SEX"]))
print(group_metrics(y_val, pred_val, sens_val["race_eth"]))
print(group_metrics(y_val, pred_val, sens_val[["race_eth", "SEX"]]))   # intersections

       Count  Selection Rate  True Positive Rate  False Positive Rate  \
SEX                                                                     
1    13576.0        0.385460            0.747613             0.150577   
2    11832.0        0.170808            0.545252             0.060311   

     Accuracy  
SEX            
1    0.809369  
2    0.849814  
            Count  Selection Rate  True Positive Rate  False Positive Rate  \
race_eth                                                                     
Hispanic   7852.0        0.101630            0.422005             0.036475   
NH Asian   1089.0        0.452709            0.830835             0.168810   
NH Black   2419.0        0.160810            0.523629             0.059259   
NH Other    447.0        0.241611            0.640000             0.086957   
NH White  13601.0        0.401882            0.743961             0.163255   

          Accuracy  
race_eth            
Hispanic  0.872007  
NH Asian  0.831038  
NH Black  0.